# Milestone2 • Colab GPU worker
Select **Runtime → Change runtime type → GPU**, then Run all. Drive authorization is interactive. Free GPU availability is not guaranteed. Use a separate job folder for each worker; never run two workers against the same output.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
JOB = '/content/drive/MyDrive/Milestone2/jobs/book-test'  # change to your uploaded job folder


In [ ]:
import pathlib, subprocess, urllib.request, hashlib, tarfile, json, shutil
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
root=Path('/content/milestone2')
revision='87ac13c6cd6c47dbf1e6e93d5ec6f4dc13bf5902'  # verified cache renderer revision
if not root.exists(): subprocess.run(['git','clone','https://github.com/davlug3/milestone2.git',str(root)],check=True)
subprocess.run(['git','fetch','origin',revision],cwd=root,check=True)
subprocess.run(['git','checkout','--detach',revision],cwd=root,check=True)
base='https://download.blender.org/release/Blender5.0/'
name='blender-5.0.1-linux-x64.tar.xz'
archive=Path('/content')/name
exe=Path('/content/blender-5.0.1-linux-x64/blender')
if not exe.is_file():
    urllib.request.urlretrieve(base+name,archive)
    expected=next(line.split()[0] for line in urllib.request.urlopen(base+'blender-5.0.1.sha256').read().decode().splitlines() if line.endswith(name))
    with archive.open('rb') as f: assert hashlib.file_digest(f,'sha256').hexdigest()==expected, 'Blender checksum mismatch'
    subprocess.run(['tar','-xJf',str(archive),'-C','/content'],check=True)
subprocess.run([str(exe),'-b','--python-exit-code','1','--python-expr','import bpy; p=bpy.context.preferences.addons["cycles"].preferences; p.compute_device_type="CUDA"; p.refresh_devices(); print([(d.name,d.type) for d in p.devices]); assert any(d.type=="CUDA" for d in p.devices)'],check=True)


In [ ]:
job=Path(JOB)
cfg=json.loads((job/'job.json').read_text())
assert cfg['version']==1 and cfg['format']=='EXR' and cfg['device']=='CUDA'
cache=job/'cache'
meta=json.loads((cache/'manifest.json').read_text())
assert 0<=cfg['first']<=cfg['last']<meta['count']
assert all(cfg[k]==meta['settings'][k] for k in ('scene','width','height','fps'))
for name,digest in cfg['hashes'].items():
    assert Path(name).name==name
    with (cache/name).open('rb') as f: assert hashlib.file_digest(f,'sha256').hexdigest()==digest, name
out=job/'frames'; out.mkdir(exist_ok=True)
marker=out/'job-settings.json'
signature=json.dumps(cfg,sort_keys=True)
if marker.exists(): assert marker.read_text()==signature, 'Use a new output folder for changed settings'
else: marker.write_text(signature)
print('Verified',cfg['scene'],cfg['first'],cfg['last'])


In [ ]:
# The existing renderer replays checkpoints and skips completed EXRs.
indices=','.join(str(i) for i in range(cfg['first'],cfg['last']+1))
cmd=[str(exe),'-b','--factory-startup','--python-exit-code','1','-P',str(root/'blender/render_cache.py'),'--','--cache',str(cache),'--output',str(out),'--format','EXR','--device','CUDA','--benchmark-frames',indices]
for key,flag in [('width','width'),('height','height'),('samples','samples'),('passes','passes'),('noise','noise'),('bounces','bounces'),('min_samples','min-samples'),('diffuse','diffuse'),('glossy','glossy'),('codec','exr-codec')]: cmd += ['--'+flag,str(cfg[key])]
subprocess.run(cmd,check=True)
missing=[i for i in range(cfg['first'],cfg['last']+1) if not (out/f'frame-{i:06d}.exr').is_file() or (out/f'frame-{i:06d}.exr').stat().st_size<=100]
assert not missing, missing
print('Requested frames complete:',out)
